# Kisara — Gambar dengan karakter konsisten (SDXL + IP-Adapter)

Notebook ini menggambar semua gambar klip **dengan wajah tokoh yang sama di setiap adegan**.
Caranya: potret tiap karakter dari Kisara dipakai sebagai acuan wajah lewat **IP-Adapter**.

**Cara pakai**
1. Di Kisara, buka proyek → langkah **Storyboard**.
2. Kalau tokohnya belum punya potret, klik **Generate gambar** dulu — potret ikut dibuat.
3. Klik **Ekspor prompt** → simpan file `...-prompts.json`.
   (File itu sudah membawa prompt **dan** potret semua karakter.)
4. Di Colab: **Runtime → Change runtime type → T4 GPU**, lalu **Save**.
5. Jalankan sel dari atas ke bawah, unggah file JSON saat diminta.
6. Unduh ZIP hasilnya, lalu di Kisara pakai tombol **Impor gambar** per klip.

> **Batasnya yang jujur:** IP-Adapter mengikat **satu** acuan wajah per gambar. Untuk adegan dengan
> satu tokoh utama, hasilnya bagus. Untuk adegan dengan dua tokoh yang sama pentingnya, wajah yang
> terikat adalah tokoh pertama — jadi kalau ada tokoh yang paling penting, taruh dia paling depan
> di daftar pemeran.
>
> Notebook ini dipakai sebagai notebook interaktif — itu cara pakai Colab yang diperbolehkan.

## 1. Pastikan dapat GPU
Kalau tidak muncul `Tesla T4`, ubah runtime ke GPU lewat menu Runtime.

In [ ]:
!nvidia-smi

## 2. Pasang dependensi

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate safetensors

## 3. Unggah file dari Kisara
Pilih file `...-prompts.json` hasil tombol **Ekspor prompt**.

In [ ]:
import base64
import io
import json
from pathlib import Path

from google.colab import files
from PIL import Image

uploaded = files.upload()
name = next(iter(uploaded))
data = json.loads(Path(name).read_text(encoding='utf-8'))

ASPECT = data.get('aspect', '16:9')
JOBS = data['images']
CHARS = {c['name']: c for c in data.get('characters', [])}


def portrait(who):
    """Potret seorang tokoh sebagai gambar PIL, atau None kalau belum digambar."""
    c = CHARS.get(who)
    if not c or not c.get('portrait'):
        return None
    raw = c['portrait'].split(',', 1)[-1]
    return Image.open(io.BytesIO(base64.b64decode(raw))).convert('RGB')


print(f"Proyek: {data.get('project', '(tanpa judul)')} · rasio {ASPECT}")
print(f'{len(JOBS)} gambar · {len(CHARS)} karakter')
for who, c in CHARS.items():
    print(f"  {who:18} potret: {'ada' if c.get('portrait') else 'BELUM ADA'}")

## 4. Atur kekuatan acuan wajah

- **`IP_SCALE`** — seberapa kuat wajah dari potret mengikat gambar. `0` = diabaikan,
  `1` = paling kuat. **0,6–0,75** biasanya paling pas: tokohnya tetap dikenali, tapi adegannya
  masih bebas bergerak. Naikkan kalau wajahnya masih berubah-ubah, turunkan kalau semua adegan
  jadi mirip potretnya.

In [ ]:
OUT_DIR = Path('/content/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

IP_SCALE = 0.7
STEPS = 30
GUIDANCE = 7.0

if ASPECT == '9:16':
    WIDTH, HEIGHT = 768, 1344
else:
    WIDTH, HEIGHT = 1344, 768

NEGATIVE = (
    '色调艳丽, 过曝, 静态, 细节模糊不清, 字幕, 风格, 作品, 画作, 画面, 静止, 整体发灰, 最差质量, 低质量, '
    'JPEG压缩残留, 丑陋的, 残缺的, 多余的手指, 画得不好的手部, 画得不好的脸部, 畸形的, 毁容的, '
    '形态畸形的肢体, 手指融合, 静止不动的画面, 杂乱的背景, 三条腿, 背景人很多, 倒着走, '
    'deformed, ugly, extra limbs, bad anatomy, worst quality, low quality, text, watermark'
)

print(f'{WIDTH}x{HEIGHT} · {STEPS} langkah · skala potret {IP_SCALE}')

## 5. Muat model
SDXL (fp16) muat nyaman di T4 16 GB. IP-Adapter hanya ~100 MB, dan itu bagian yang mengikat wajah.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16
)
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models', weight_name='ip-adapter_sdxl.bin')
pipe.set_ip_adapter_scale(IP_SCALE)
pipe.to('cuda')
print('model siap')

## 6. Gambar semuanya
Gambar yang sudah ada dilewati, jadi aman dijalankan ulang kalau sesi Colab terputus.

In [ ]:
import gc
import time

done = []
for job in JOBS:
    out = OUT_DIR / f"{job['index']:03d}.png"
    if out.exists():
        print(f"[{job['index']}/{len(JOBS)}] lewati (sudah ada)")
        done.append(out)
        continue

    # Adegan dengan beberapa tokoh diikat ke tokoh pertama yang punya potret.
    ref = None
    ref_name = None
    for who in job.get('characters', []):
        ref = portrait(who)
        if ref is not None:
            ref_name = who
            break

    note = f' (acuan wajah: {ref_name})' if ref is not None else ''
    print(f"[{job['index']}/{len(JOBS)}] menggambar{note}")
    started = time.time()
    kwargs = {'ip_adapter_image': ref} if ref is not None else {}
    image = pipe(
        prompt=job['prompt'],
        negative_prompt=NEGATIVE,
        num_inference_steps=STEPS,
        guidance_scale=GUIDANCE,
        width=WIDTH,
        height=HEIGHT,
        **kwargs,
    ).images[0]
    image.save(out)
    done.append(out)
    print(f'    selesai dalam {time.time() - started:.0f} detik')
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nSelesai: {len(done)} gambar di {OUT_DIR}')

## 7. Unduh hasilnya
Ekstrak ZIP-nya, lalu di Kisara buka tiap klip → **Impor gambar** dan pilih file sesuai nomornya.

In [ ]:
import os
import shutil
from google.colab import files

zip_path = shutil.make_archive('/content/kisara-consistent', 'zip', OUT_DIR)
print('Ukuran:', round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
files.download(zip_path)

## Kalau ada masalah

**Wajahnya masih berubah-ubah antar adegan** — naikkan `IP_SCALE` (misalnya 0,8), lalu jalankan
ulang sel 4–6. Pastikan juga potret tokohnya sudah ada (lihat cetakan di sel 3).

**Semua adegan jadi mirip potretnya** — turunkan `IP_SCALE` (misalnya 0,55). Kalau terlalu kuat,
model mempertahankan latar polos dari potret juga.

**Tokoh tidak dikenali sama sekali** — potretnya mungkin belum dibuat. Di Kisara, klik
**Generate gambar**; potret ikut dibuat dan muncul di panel pemeran.

**CUDA out of memory** — ganti `pipe.to('cuda')` di sel 5 menjadi `pipe.enable_model_cpu_offload()`,
lalu jalankan ulang sel 5 dan 6. Lebih lambat, tapi jauh lebih hemat memori.

**Sesi Colab mati di tengah jalan** — jalankan ulang sel 4–6. Gambar yang sudah jadi dilewati.

**Adegan dengan dua tokoh wajahnya tercampur** — itu batas IP-Adapter: satu gambar hanya bisa
diikat ke satu acuan wajah. Pecah adegan itu jadi dua klip, atau taruh tokoh terpenting lebih
dulu di daftar pemeran proyek.